# 1. 비만 데이터 전처리 (Preprocessing)

**목적:** 원본 데이터를 로드하고 EDA를 수행한 뒤, 이후 노트북에서 재사용 가능한
인코더(`encoders.pkl`)와 스케일러(`scaler.pkl`)를 저장합니다.

**핵심 설계 원칙 – 데이터 누수 방지:**
- `Height`, `Weight`는 타겟 `NObeyesdad`의 직접 파생 변수(BMI = Weight/Height²)입니다.
- 이 두 컬럼을 피처로 사용하면 모델이 사실상 BMI 공식을 재학습하는 것과 같아
  99%+ 정확도가 나오지만 **실제 예측 능력은 없습니다**.
- 재작성된 코드는 **행동·생활습관 피처만** 사용합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

os.makedirs('../artifacts', exist_ok=True)

DATA_PATH = '../data/obesity.csv'
ARTIFACTS = '../artifacts'

In [ ]:
data = pd.read_csv(DATA_PATH)
print(f'Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
data.head()

## 타겟 분포 확인

In [ ]:
obesity_order = [
    'Insufficient_Weight', 'Normal_Weight',
    'Overweight_Level_I', 'Overweight_Level_II',
    'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III'
]

counts = data['NObeyesdad'].value_counts().reindex(obesity_order)
plt.figure(figsize=(10, 4))
sns.barplot(x=obesity_order, y=counts.values, palette='RdYlGn_r')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.title('Obesity Level Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(counts)

## 피처 선택

행동·생활습관 피처만 사용하고 `Height`, `Weight`는 제외합니다.

In [ ]:
CATEGORICAL_FEATURES = [
    'Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE',
    'family_history_with_overweight', 'CAEC', 'MTRANS'
]
CONTINUOUS_FEATURES = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
TARGET = 'NObeyesdad'

print('피처 (behavioral only):')
print(f'  범주형: {CATEGORICAL_FEATURES}')
print(f'  연속형: {CONTINUOUS_FEATURES}')
print(f'  제외 (data leakage): [Height, Weight]')

## EDA – 연속형 피처 분포

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), CONTINUOUS_FEATURES):
    sns.histplot(data[col], kde=True, ax=ax, bins=25, color='steelblue')
    ax.set_title(col)
plt.suptitle('Behavioral Continuous Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## EDA – 범주형 피처 분포

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), CATEGORICAL_FEATURES):
    vc = data[col].value_counts()
    sns.barplot(x=vc.index, y=vc.values, ax=ax, palette='Set2')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 범주형 인코딩

- **이진 피처**: 딕셔너리 매핑으로 0/1
- **순서형 피처**: 순서가 있는 카테고리를 정수로 (CALC, CAEC: no < Sometimes < Frequently < Always)
- 인코더 딕셔너리를 `artifacts/encoders.pkl`에 저장 → 예측 시 동일 매핑 재사용

In [ ]:
BINARY_MAP = {
    'Gender': {'Female': 0, 'Male': 1},
    'FAVC':   {'no': 0, 'yes': 1},
    'SCC':    {'no': 0, 'yes': 1},
    'SMOKE':  {'no': 0, 'yes': 1},
    'family_history_with_overweight': {'no': 0, 'yes': 1},
}
ORDINAL_MAP = {
    'CALC':   ['no', 'Sometimes', 'Frequently', 'Always'],
    'CAEC':   ['no', 'Sometimes', 'Frequently', 'Always'],
    'MTRANS': ['Bike', 'Walking', 'Public_Transportation', 'Motorbike', 'Automobile'],
}

df = data.copy()

for col, mapping in BINARY_MAP.items():
    df[col] = df[col].map(mapping)

for col, order in ORDINAL_MAP.items():
    order_map = {v: i for i, v in enumerate(order)}
    df[col] = df[col].map(order_map)

encoders = {'binary': BINARY_MAP, 'ordinal': ORDINAL_MAP}
with open(f'{ARTIFACTS}/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print('encoders.pkl 저장 완료')
df[CATEGORICAL_FEATURES].head()

## 연속형 피처 스케일링

StandardScaler를 피팅하고 `artifacts/scaler.pkl`에 저장합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[CONTINUOUS_FEATURES] = scaler.fit_transform(df[CONTINUOUS_FEATURES])

with open(f'{ARTIFACTS}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('scaler.pkl 저장 완료')
print(f'스케일링 전 평균(원본): {data[CONTINUOUS_FEATURES].mean().round(2).to_dict()}')
print(f'스케일링 후 평균:       {df[CONTINUOUS_FEATURES].mean().round(4).to_dict()}')

## 상관관계 분석

In [ ]:
feature_df = df[CATEGORICAL_FEATURES + CONTINUOUS_FEATURES]
corr = feature_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.4)
plt.title('Feature Correlation Heatmap (Behavioral Features)')
plt.tight_layout()
plt.show()

## 전처리 완료 데이터 저장

In [ ]:
save_cols = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES + [TARGET]
df[save_cols].to_csv(f'{ARTIFACTS}/processed_data.csv', index=False)

print(f'processed_data.csv 저장 완료: {df.shape}')
print(f'\n피처 ({len(save_cols)-1}개) + 타겟 저장')
df[save_cols].head()